# Install Library

In [1]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Library

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from google.colab import drive

# Connect to Drive

In [4]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Path folder tempat file CSV berada
combined_df = pd.read_csv('/content/drive/MyDrive/File/CODES/Similarity/7_Similarity_Nonrelevan_Tambahan_Base1/pre_nonrelevan.csv')

combined_df.head()

,title,content
0,Salmafina Sunan Ungkap Sunan Kalijaga Sudah Bi...,"Meski sempat menghilang dan membuat,khawatir, ..."
1,Satria Mulia Oplas Mirip Jimin BTS?,"Nama,mendadak jadi perbincangan fans K-Pop, kh..."
2,Meghan Markle Santai di Acara Olahraga dengan ...,mengejutkan banyak mata dengan kemunculannya d...
3,Mpok Alpa Berbagi Kisah Masa Kecil di Kampung ...,Nama Mpok Alpa mulai dikenal ketika videonya m...
4,Admin Lambe Turah Kenang Perjalanan Karier di ...,"YouTuber,mendapat kesempatan mendatangi salah ..."


In [7]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2967 entries, 0 to 2966
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    2967 non-null   object
 1   content  2963 non-null   object
dtypes: object(2)
memory usage: 46.5+ KB


In [8]:
nan_summary = combined_df[combined_df['content'].isna()]


nan_summary

,title,content
406,Teori Baru: Black Widow Ternyata Masih Hidup d...,NaN
1648,INFOGRAFIS: Batik Ternyata Ditemukan di Luar A...,NaN
2562,KAI Umumkan Layanan Kereta Ekspres untuk Wisat...,NaN
2770,INFOGRAFIS: Panduan Mengamankan Data Pribadi d...,NaN


In [9]:
combined_df=combined_df.dropna()

# Duplicate


In [10]:
data = combined_df.copy()

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2963 entries, 0 to 2966
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    2963 non-null   object
 1   content  2963 non-null   object
dtypes: object(2)
memory usage: 69.4+ KB


# Tokenize

Mencoba kode lain agar tokenize lebih mudah di pahami

# Word Embedding

In [12]:
from transformers import AutoTokenizer, AutoModel
import torch

# Memuat IndoBERT dan tokenizer
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1", use_fast=True)
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [ ]:
# Fungsi untuk mendapatkan embedding dari IndoBERT
def get_embeddings(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    # Mengambil hidden states dari layer terakhir dan rata-rata embeddingsnya
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk mengonversi tokenized data ke dalam bentuk input_ids dan mendapatkan embedding
def get_input_ids_and_embeddings(data_column):
    input_ids_list = []
    embeddings_list = []

    for text in data_column:
        # Tokenisasi
        tokens = tokenizer(text, padding='max_length', max_length=256, truncation=True, return_tensors="pt")
        input_ids = tokens['input_ids']  # Ambil input_ids
        input_ids_list.append(input_ids)

        # Mendapatkan embeddings
        embedding = get_embeddings(input_ids)
        embeddings_list.append(embedding)

    return input_ids_list, embeddings_list

# Dapatkan input_ids dan embeddings untuk title dan summary
title_input_ids, title_embeddings = get_input_ids_and_embeddings(data['title'])
summary_input_ids, summary_embeddings = get_input_ids_and_embeddings(data['content'])


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


## Mengubah embeddings menjadi array numpy

In [ ]:
# Mengubah embeddings menjadi array numpy agar bisa disimpan dalam DataFrame
title_embeddings_array = np.vstack([embedding.numpy() for embedding in title_embeddings])
summary_embeddings_array = np.vstack([embedding.numpy() for embedding in summary_embeddings])

# Membuat DataFrame baru dengan embeddings numerik
data_embeddings = pd.DataFrame({
    'title_embeddings': list(title_embeddings_array),
    'summary_embeddings': list(summary_embeddings_array)
})

# Cosine Similarity

In [ ]:
# hitung cosine similarity pada matrix berita
cosine_sim = cosine_similarity(title_embeddings_array, summary_embeddings_array)
print(cosine_sim)

[[0.5278511  0.27399862 0.16771808 ... 0.13188232 0.12215213 0.25776255]
 [0.5217369  0.27286768 0.16877824 ... 0.13374084 0.12523636 0.25760067]
 [0.5229429  0.27306226 0.17305252 ... 0.13382775 0.12758663 0.2620887 ]
 ...
 [0.5214254  0.2743776  0.16847709 ... 0.13793203 0.12642518 0.25872248]
 [0.52287346 0.27379805 0.16911002 ... 0.13457265 0.12809786 0.25923714]
 [0.52326506 0.27314982 0.16869836 ... 0.13468611 0.1257148  0.261435  ]]


In [ ]:
# membuat dataframe dari varible cosine similarity dengan baris dan kolom title
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['content'], columns=data['title'])

print(cosine_sim_df.shape)

(2963, 2963)


## Check Similarity Result

In [ ]:
# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # The index is a summary, and you need to find the corresponding title
    # to access the similarity value. However, since you have both summaries
    # and titles as indices, you can access the similarity directly using
    # the index and the corresponding title from the 'data' DataFrame.

    title = data.loc[data['content'] == index, 'title'].iloc[0] # Get the title for the current summary
    similarity_value = cosine_sim_df.loc[index, title]  # Access the similarity using both summary and title

    print(f"Title: {title}")
    print(f"Summary: {index}")
    print(f"Cosine Similarity: {similarity_value}\n")
    print("-" * 50)  # Pemisah antar pasangan

Streaming output truncated to the last 5000 lines.
Title: Adegan Ciuman di Drama Ini Jadi Viral, Tapi Bukan Karena Keromantisannya!
Summary: "," menayangkan episode terbaru pada Sabtu (28/9). Di episode ke-4 itu, drama SBS ini menayangkan sebuah adegan mengejutkan antara Go Hae Ri (,) dan Ki Tae Woong (,).,,,Sebelum dipindahkan ke Maroko, Go Hae Ri rupanya memiliki perasaan suka pada Ki Tae Woong. Dalam keadaan mabuk saat nongkrong bersama tim NIS, Go Hae Ri dengan lantang menyatakan kepemilikannya atas Ki Tae Woong.,Menariknya, Go Hae Ri dengan berani mencium bibir Ki Tae Woong. Tingkah tak sadar agen cantik itu tentunya membuat tim NIS yang lain kaget termasuk Ki Tae Woong sendiri yang tampak hampir marah.,Adegan ciuman pertama di "Vagabond" antara Suzy dan Shin Sung Rok ini tentu saja membuat penonton heboh. Banyak yang menyebut Shin Sung Rok sangat beruntung karena berhasil mendapatkan ciuman Suzy.,,,"," tulis seorang fans. ",," sambung yang lain.,",," sahut fans lain. ",," pungkas

In [ ]:
# List to store the results
results = []

# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # Mendapatkan judul yang sesuai dengan summary berdasarkan index
    title = data.loc[data['content'] == index, 'title'].iloc[0]
    similarity_value = cosine_sim_df.loc[index, title]  # Mengakses nilai similarity

    # Menyimpan pasangan dan nilai similarity ke dalam list
    results.append([title, index, similarity_value])

# Membuat DataFrame untuk hasilnya
similarity_df = pd.DataFrame(results, columns=['Title', 'Summary', 'Cosine Similarity'])


In [ ]:
similarity_df.head()

,Title,Summary,Cosine Similarity
0,Salmafina Sunan Ungkap Sunan Kalijaga Sudah Bi...,"Meski sempat menghilang dan membuat,khawatir, ...",0.527851
1,Satria Mulia Oplas Mirip Jimin BTS?,"Nama,mendadak jadi perbincangan fans K-Pop, kh...",0.272868
2,Meghan Markle Santai di Acara Olahraga dengan ...,mengejutkan banyak mata dengan kemunculannya d...,0.173053
3,Mpok Alpa Berbagi Kisah Masa Kecil di Kampung ...,Nama Mpok Alpa mulai dikenal ketika videonya m...,0.161155
4,Admin Lambe Turah Kenang Perjalanan Karier di ...,"YouTuber,mendapat kesempatan mendatangi salah ...",0.124442


## Save Similarity

In [ ]:
# Menyimpan DataFrame ke file CSV di lokasi yang ditentukan
similarity_df.to_csv('/content/drive/MyDrive/File/CODES/Similarity/7_Similarity_Nonrelevan_Tambahan_Base1/similarity_nonrelevan_tambah_base1.csv', index=False)